# 8-2 실습 예제 · DAG 이해 및 유즈케이스 연구

**데이터 엔지니어링 & RAG 파이프라인 · 8주차 2일차** — 수업에서 표로만 본 유즈케이스를 **실제로 도는 DAG** 로 만들고, 심화 과제의 문법 네 가지를 풀이와 함께 돌려 봅니다.

- 걸리는 시간: 약 60분
- 키: 필요 없습니다(전부 키 없이 끝까지 돕니다).

### 시작하는 방법
1. 「Drive로 복사」를 누른 뒤 위에서부터 한 칸씩 실행합니다(칸 왼쪽 ▶ 또는 Shift+Enter).
2. 준비 칸 둘(설치 · 도구)은 처음 한 번만 실행합니다. 설치는 1~2분 걸립니다.
3. 예제마다 **확인 포인트**와 **바꿔 보기**가 있습니다. 확인 포인트의 값이 화면과 같은지 보고, 바꿔 보기는 직접 고쳐 다시 실행합니다.

### 수업 환경과 다른 점
수업에서는 `docker compose` 로 일곱 컨테이너를 띄웠습니다. 여기서는 같은 판(Airflow 3.3.2)을 코랩 안 가상환경 하나에 넣어, 설치 걱정 없이 DAG 에만 집중합니다. 코랩 런타임이 끊기면 처음부터 다시 실행하면 됩니다.

### 예제 구성
| 번호 | 예제 | 수업 노트와 이어지는 곳 |
|---|---|---|
| 1 | 야간 정산 — 태스크끼리 값 넘기기(XCom) | 유즈케이스 표 1행 · 심화 과제 4 |
| 2 | 데이터 품질 점검 — 실패로 막기와 건너뛰기 | 유즈케이스 표 4행 · 실습 6단계(실패 전파) |
| 3 | 모델 재학습 — 점수에 따라 갈라지기 | 유즈케이스 표 2행 |
| 4 | 리포트 발송 — 집계를 기다렸다 합류, 실패하면 알림 | 유즈케이스 표 3행 · 실습 8단계(병렬과 합류) |
| 5 | TaskFlow — 같은 정산을 @dag · @task 로 | 심화 과제 3 |
| 6 | 스케줄 — 크론 표기와 다음 실행 시각 | 이론 「스케줄」 · 심화 과제 2 |
| 7 | catchup — 밀린 날짜가 한꺼번에 도는 것 | 이론 「스케줄」 |
| 8 | 재시도와 멱등성 — 다시 돌려도 안전한가 | 이론 「멱등성」 · 「재시도 설정」 |

## 준비

In [ ]:
# 준비 1 — Airflow 3.3.2 설치 (1~2분, 처음 한 번만)
import os, subprocess, sys, time

os.environ["TZ"] = "Asia/Seoul"          # 코랩 시계는 UTC — 화면의 시각을 한국 기준으로
time.tzset()

AF_VER = "3.3.2"                          # 수업과 같은 판
PY = f"{sys.version_info.major}.{sys.version_info.minor}"
HOME = "/content"
VENV = f"{HOME}/af"                       # 코랩 꾸러미와 섞이지 않게 가상환경에 설치합니다

os.environ.update({
    "AIRFLOW_HOME": f"{HOME}/airflow",
    "AIRFLOW__CORE__DAGS_FOLDER": f"{HOME}/dags",      # 수업의 ~/airflow-lab/dags 와 같은 자리
    "AIRFLOW__CORE__LOAD_EXAMPLES": "False",
    "AIRFLOW__CORE__PARALLELISM": "4",
    "NO_COLOR": "1",                                    # 로그에 색 코드가 섞이지 않게
    "PATH": f"{VENV}/bin:" + os.environ["PATH"],
})
for d in ("dags", "data"):
    os.makedirs(f"{HOME}/{d}", exist_ok=True)

if not os.path.exists(f"{VENV}/bin/airflow"):
    print("설치 중입니다 …")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True, capture_output=True)
    subprocess.run([sys.executable, "-m", "uv", "venv", "-q", VENV, "--python", sys.executable], check=True)
    r = subprocess.run(
        [sys.executable, "-m", "uv", "pip", "install", "-q", "--python", f"{VENV}/bin/python",
         f"apache-airflow=={AF_VER}", "graphviz",
         "--constraint", f"https://raw.githubusercontent.com/apache/airflow/constraints-{AF_VER}/constraints-{PY}.txt"],
        capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])
        raise SystemExit("설치하지 못했습니다. 이 칸을 한 번 더 실행해 주세요.")

r = subprocess.run("airflow db migrate", shell=True, capture_output=True, text=True)   # 메타데이터 DB(SQLite) 만들기
v = subprocess.run("airflow version", shell=True, capture_output=True, text=True).stdout.strip().splitlines()[-1]
print("설치 확인 · Airflow", v, "· 파이썬", PY)
print("DAG 폴더 ·", os.environ["AIRFLOW__CORE__DAGS_FOLDER"])

In [ ]:
# 준비 2 — 이 노트북에서 쓰는 도구 셋 (처음 한 번만)
import json, re
from datetime import datetime, timezone
from IPython.display import Image, display

_LOG = re.compile(r"^\d{4}-\d\d-\d\dT\S+ \[")
_ANSI = re.compile(r"\x1b\[[0-9;]*m")


def _sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return _ANSI.sub("", r.stdout + r.stderr)
_NOISE = ("graphviz", "You might need to install", "Please be reminded", "DagRun failed")


def af(args, show=True):
    """수업의 af 함수와 같습니다 — af("dags list") = airflow dags list. 로그 줄은 빼고 결과만 보여 줍니다."""
    keep = [l for l in _sh("airflow " + args).splitlines()
            if l.strip() and not _LOG.match(l) and not any(n in l for n in _NOISE)]
    text = "\n".join(keep)
    if show:
        print(text)
    return text


def graph(dag_id):
    """DAG 모양을 그림으로 — 웹 UI 의 Graph 화면과 같은 것입니다."""
    png = f"{HOME}/data/{dag_id}.png"
    if os.path.exists(png):
        os.remove(png)
    af(f"dags show {dag_id} --save {png}", show=False)
    if not os.path.exists(png):             # 아직 등록 전인 새 파일이면 먼저 읽게 한 뒤 다시
        af("dags reserialize", show=False)
        af(f"dags show {dag_id} --save {png}", show=False)
    if os.path.exists(png):
        display(Image(png))
    else:                                   # 그림 도구가 없으면 화살표만 글자로
        print("\n".join(l.strip() for l in af(f"dags show {dag_id}", show=False).splitlines() if "->" in l))


def _utc(s):
    return datetime.fromisoformat(s).replace(tzinfo=timezone.utc)      # DB 의 시각은 UTC


def _kst(s):
    return _utc(s).astimezone().strftime("%H:%M:%S.%f")[:-3] if s else "-"


def run_dag(dag_id, conf=None):
    """DAG 를 한 번 끝까지 돌리고(airflow dags test), 태스크가 찍은 글과 태스크별 상태를 보여 줍니다."""
    cmd = f"airflow dags test {dag_id}" + (f" -c '{json.dumps(conf)}'" if conf else "")
    print(f"$ {cmd}\n실행 중입니다 …\n")
    out = _sh(cmd)
    task, tb = "-", []

    def flush():
        if tb:
            print(f"  [{task}] 오류 → {tb[-1].strip()}")
            tb.clear()

    for l in out.splitlines():
        m = re.search(r"\[DAG TEST\] starting task_id=(\S+)", l)
        if m:
            flush()
            task = m.group(1)
            continue
        if _LOG.match(l):
            flush()
            m = re.search(r"Done\. Returned value was: (.*?)\s+\[airflow", l)
            if m and m.group(1) != "None":
                print(f"  [{task}] 돌려준 값(XCom) → {m.group(1)}")
            m = re.search(r"Branch into (\S+)", l)
            if m:
                print(f"  [{task}] 고른 갈래 → {m.group(1)}")
            if "Skipping downstream tasks" in l and "ShortCircuitOperator" in l:
                print(f"  [{task}] False 를 돌려줘 뒤 태스크를 건너뜁니다")
            continue
        if not l.strip() or any(n in l for n in _NOISE):
            continue
        if tb or l.startswith("Traceback"):
            tb.append(l)
        else:
            print(f"  [{task}] {l}")
    flush()

    m = re.search(r"run_id=(manual__[^\s,]+)", out)
    if not m:                               # DAG 를 읽지 못한 경우 — 원인을 그대로 보여 줍니다
        print("\n".join(l for l in out.splitlines() if not _LOG.match(l))[-1500:])
        print("\nDAG 를 돌리지 못했습니다. 위 오류와 af(\"dags list-import-errors\") 를 확인해 주세요.")
        return {}
    import sqlite3                           # 태스크 상태는 메타데이터 DB 에서 바로 읽습니다(웹 UI 의 Grid 가 보는 곳)
    db = sqlite3.connect(f"{HOME}/airflow/airflow.db")
    rows = [dict(zip(("task_id", "state", "start_date", "end_date", "try_number"), r)) for r in db.execute(
        "select task_id, state, start_date, end_date, try_number from task_instance where dag_id=? and run_id=?",
        (dag_id, m.group(1)))]
    db.close()
    rows.sort(key=lambda x: x.get("start_date") or "9")
    print(f"\n{'태스크':<16}{'상태':<18}{'시도':<5}{'시작(한국 시각)':<16}걸린 시간")
    for x in rows:
        sec = ""
        if x.get("start_date") and x.get("end_date"):
            sec = f"{(_utc(x['end_date']) - _utc(x['start_date'])).total_seconds():.2f}초"
        print(f"{x['task_id']:<19}{(x['state'] or '-'):<20}{x['try_number']:<7}{_kst(x.get('start_date')):<20}{sec}")
    state = re.search(r"DagRun Finished:.*?state=(\w+)", out)
    print(f"\n실행 전체 상태 → {state.group(1) if state else '?'}")
    return {x["task_id"]: x["state"] for x in rows}


print("도구 준비 완료 · af( ) · graph( ) · run_dag( )")

## 예제 1 · 야간 정산 — 태스크끼리 값 넘기기 (XCom)

수업의 `etl_practice` 는 세 태스크가 `print` 만 했습니다. 실제 정산이라면 **앞 태스크가 센 건수를 뒤 태스크가 받아** 맞는지 확인해야 합니다.
Airflow 에서는 태스크 함수가 **돌려준 값(return)** 이 저절로 XCom 에 저장되고, 뒤 태스크가 `ti.xcom_pull(task_ids="앞 태스크")` 로 꺼냅니다.

- `extract` — 주문 20건을 CSV 로 만들고 **건수를 돌려줍니다**
- `summarize` — 건수를 받아 CSV 의 실제 줄 수와 맞는지 보고, 합계를 돌려줍니다
- `load` — 결과를 JSON 파일로 남깁니다

In [ ]:
%%writefile /content/dags/settle_dag.py
# settle_dag.py — 야간 정산: 수집 → 집계 → 적재, 값은 XCom 으로 넘긴다
import csv, json
from datetime import datetime
from pathlib import Path

from airflow.sdk import DAG
from airflow.providers.standard.operators.python import PythonOperator

DATA = Path("/content/data")
ROWS = 20


def extract():
    rows = [(i, 1000 * (i % 7 + 1)) for i in range(1, ROWS + 1)]
    with open(DATA / "orders.csv", "w", newline="") as f:
        csv.writer(f).writerows([("order_id", "amount"), *rows])
    print(f"주문 {len(rows)}건을 orders.csv 로 받았습니다")
    return len(rows)                        # 돌려준 값이 XCom 에 저장됩니다


def summarize(ti):
    expected = ti.xcom_pull(task_ids="extract")           # 앞 태스크가 돌려준 값
    amounts = [int(r["amount"]) for r in csv.DictReader(open(DATA / "orders.csv"))]
    if len(amounts) != expected:
        raise ValueError(f"건수가 다릅니다 — 받은 값 {expected}, 파일 {len(amounts)}")
    print(f"건수 {expected}건 일치 · 합계 {sum(amounts):,}원")
    return {"count": expected, "total": sum(amounts)}


def load(ti):
    result = ti.xcom_pull(task_ids="summarize")
    (DATA / "settle_result.json").write_text(json.dumps(result))
    print("settle_result.json 저장 →", result)


with DAG(
    dag_id="settle_dag",
    start_date=datetime(2026, 1, 1),
    schedule="0 2 * * *",                   # 매일 02:00 — 야간 정산
    catchup=False,
    tags=["week8", "usecase"],
) as dag:
    t1 = PythonOperator(task_id="extract", python_callable=extract)
    t2 = PythonOperator(task_id="summarize", python_callable=summarize)
    t3 = PythonOperator(task_id="load", python_callable=load)

    t1 >> t2 >> t3

In [ ]:
s = run_dag("settle_dag")
graph("settle_dag")

***확인 포인트***: `extract` 가 돌려준 값 20 이 `summarize` 줄의 「건수 20건 일치 · 합계 83,000원」으로 이어지고, 세 태스크가 모두 success 입니다.
`돌려준 값(XCom)` 줄이 두 번 보입니다 — `extract` 의 20 과 `summarize` 의 `{'count': 20, 'total': 83000}`.
표의 첫 태스크만 걸린 시간이 긴 것은 Airflow 가 실행을 준비하는 시간이 함께 잡혀서입니다. 표 아래 그림은 웹 UI 의 Graph 화면과 같은 모양입니다.

**바꿔 보기**
- 파일 위쪽의 `ROWS = 20` 을 `30` 으로 고쳐 다시 실행합니다. 합계는 얼마가 되나요?
- `summarize` 의 `task_ids="extract"` 를 `"extrac"` 로 틀리게 적고 다시 실행합니다. 어느 태스크가 어떤 글과 함께 실패하나요? 태스크 이름은 글자로 적는 것이라 오타를 파이썬이 미리 잡아 주지 않습니다.
- XCom 에는 작은 값(건수 · 경로 · 요약)만 넣습니다. 데이터 자체는 파일이나 버킷에 두고 **경로만** 넘깁니다 — 8-3 에서 그렇게 합니다.

## 예제 2 · 데이터 품질 점검 — 실패로 막기와 건너뛰기

유즈케이스 표의 「건수 · 널 · 중복을 검사해 기준 미달이면 다음 단계 중단」입니다. 중단하는 방법은 두 가지입니다.

| 방법 | 태스크 상태 | 언제 쓰나 |
|---|---|---|
| 검사 태스크에서 **오류를 낸다**(`raise`) | 검사 failed · 뒤 upstream_failed | 사람이 봐야 하는 사고 — 알림이 가고 재시도 대상이 된다 |
| **ShortCircuitOperator** 가 False 를 돌려준다 | 검사 success · 뒤 skipped | 「오늘은 할 일 없음」처럼 정상적인 건너뜀 |

데이터를 일부러 나쁘게 만드는 스위치(`bad`)를 `params` 로 두었습니다. 실행할 때 `conf` 로 값을 바꿉니다.

In [ ]:
%%writefile /content/dags/quality_dag.py
# quality_dag.py — 품질 점검: 미달이면 적재를 막는다
import csv
from datetime import datetime
from pathlib import Path

from airflow.sdk import DAG
from airflow.providers.standard.operators.python import PythonOperator, ShortCircuitOperator

DATA = Path("/content/data")


def make_data(params):
    rows = [(i, 1000 * (i % 7 + 1)) for i in range(1, 21)]
    if params["bad"]:                       # 일부러 나쁜 데이터: 금액 빈칸 2건 + 중복 주문 1건
        rows[3] = (4, "")
        rows[9] = (10, "")
        rows.append((20, 7000))
    with open(DATA / "q_orders.csv", "w", newline="") as f:
        csv.writer(f).writerows([("order_id", "amount"), *rows])
    print(f"{len(rows)}건 준비 · bad={params['bad']}")


def inspect():
    rows = list(csv.DictReader(open(DATA / "q_orders.csv")))
    nulls = sum(1 for r in rows if r["amount"] == "")
    dups = len(rows) - len({r["order_id"] for r in rows})
    print(f"건수 {len(rows)} · 빈 금액 {nulls} · 중복 주문 {dups}")
    return nulls, dups


def check_or_fail():
    nulls, dups = inspect()
    if nulls or dups:
        raise ValueError(f"품질 미달 — 빈 금액 {nulls}건, 중복 {dups}건")


def check_or_skip():
    nulls, dups = inspect()
    return nulls == 0 and dups == 0          # False 면 뒤 태스크를 건너뜁니다


def load():
    print("적재 완료")


for name, checker in [("quality_fail_dag", "fail"), ("quality_skip_dag", "skip")]:
    with DAG(
        dag_id=name,
        start_date=datetime(2026, 1, 1),
        schedule=None,
        catchup=False,
        params={"bad": False},
        tags=["week8", "usecase"],
    ) as dag:
        make = PythonOperator(task_id="make_data", python_callable=make_data)
        if checker == "fail":
            check = PythonOperator(task_id="check", python_callable=check_or_fail)
        else:
            check = ShortCircuitOperator(task_id="check", python_callable=check_or_skip)
        make >> check >> PythonOperator(task_id="load", python_callable=load)
    globals()[name] = dag                    # 한 파일에 DAG 가 둘 — 둘 다 등록되게

먼저 깨끗한 데이터로 돌려 대조군을 만듭니다.

In [ ]:
s = run_dag("quality_fail_dag")

이제 나쁜 데이터를 넣습니다. **오류로 막는 쪽**부터 봅니다.

In [ ]:
s = run_dag("quality_fail_dag", conf={"bad": True})

같은 나쁜 데이터를 **건너뛰는 쪽**에 넣습니다.

In [ ]:
s = run_dag("quality_skip_dag", conf={"bad": True})

***확인 포인트***: 세 번의 실행을 나란히 봅니다.

| 실행 | check | load | 실행 전체 |
|---|---|---|---|
| 깨끗한 데이터 | success | success | success |
| 나쁜 데이터 · 오류로 막기 | **failed** — 「품질 미달 — 빈 금액 2건, 중복 1건」 | **upstream_failed** | failed |
| 나쁜 데이터 · 건너뛰기 | success | **skipped** | success |

두 번 모두 나쁜 데이터는 적재되지 않았습니다. 다른 것은 **실행 전체 상태**입니다. 오류로 막으면 실행이 빨갛게 남아 누군가 보게 되고, 건너뛰면 초록이라 아무도 모릅니다.
8-3 에서 만드는 `validate` 태스크는 그래서 오류로 막는 쪽을 씁니다.

**바꿔 보기**
- 기준을 느슨하게 바꿔 봅니다. `check_or_fail` 에서 「빈 금액이 전체의 10% 이하면 통과」로 고치면 같은 나쁜 데이터가 통과하나요? (21건 중 2건)
- `make_data` 가 빈 파일을 만들면(0건) 지금 검사는 통과합니다. **건수 하한** 검사를 한 줄 더해 막아 봅니다.

## 예제 3 · 모델 재학습 — 점수에 따라 갈라지기

유즈케이스 표의 「데이터 준비 → 학습 → 평가 → 배포를 **조건부로** 진행」입니다. 평가 점수가 기준(0.90) 이상이면 배포하고, 아니면 보류합니다.
`BranchPythonOperator` 는 함수가 **돌려준 task_id 쪽만** 실행하고 나머지 갈래는 skipped 로 만듭니다.

여기에는 함정이 하나 있습니다. 먼저 그대로 돌려 직접 봅니다.

In [ ]:
%%writefile /content/dags/retrain_dag.py
# retrain_dag.py — 모델 재학습: 평가 점수에 따라 배포 또는 보류
from datetime import datetime

from airflow.sdk import DAG
from airflow.providers.standard.operators.python import PythonOperator, BranchPythonOperator

THRESHOLD = 0.90


def prepare():
    print("학습 데이터 준비")


def train(params):
    print(f"학습 끝 · 평가 점수 {params['score']}")
    return params["score"]                  # 실제라면 평가 결과가 여기서 나옵니다


def decide(ti):
    score = ti.xcom_pull(task_ids="train")
    return "deploy" if score >= THRESHOLD else "hold"      # 실행할 task_id 를 돌려줍니다


def deploy():
    print("새 모델을 배포합니다")


def hold():
    print("기준 미달 — 기존 모델을 유지합니다")


def notify():
    print("결과를 담당자에게 알립니다")


with DAG(
    dag_id="retrain_dag",
    start_date=datetime(2026, 1, 1),
    schedule="@weekly",
    catchup=False,
    params={"score": 0.93},
    tags=["week8", "usecase"],
) as dag:
    t_prepare = PythonOperator(task_id="prepare", python_callable=prepare)
    t_train = PythonOperator(task_id="train", python_callable=train)
    t_decide = BranchPythonOperator(task_id="decide", python_callable=decide)
    t_deploy = PythonOperator(task_id="deploy", python_callable=deploy)
    t_hold = PythonOperator(task_id="hold", python_callable=hold)
    t_notify = PythonOperator(task_id="notify", python_callable=notify)

    t_prepare >> t_train >> t_decide >> [t_deploy, t_hold] >> t_notify

In [ ]:
s = run_dag("retrain_dag")
graph("retrain_dag")

***확인 포인트***: 점수 0.93 이라 `decide` 가 `deploy` 갈래를 골랐고 `hold` 는 skipped 입니다. 여기까지는 뜻대로입니다.
그런데 **`notify` 도 skipped** 입니다. 배포했는데 알림이 가지 않았습니다.

이유는 기본 규칙에 있습니다. 태스크는 **앞 태스크가 전부 success 일 때만** 돕니다(`trigger_rule="all_success"`). `hold` 가 skipped 라 `notify` 도 건너뛴 것입니다.
갈래가 합류하는 태스크에는 「실패한 것이 없고 하나라도 성공했으면 실행」 규칙을 줍니다.

In [ ]:
# notify 에 trigger_rule 을 더해 파일을 고칩니다
from pathlib import Path
p = Path("/content/dags/retrain_dag.py")
old = 'PythonOperator(task_id="notify", python_callable=notify)'
new = 'PythonOperator(task_id="notify", python_callable=notify, trigger_rule="none_failed_min_one_success")'
p.write_text(p.read_text().replace(old, new))
print([l.strip() for l in p.read_text().splitlines() if "trigger_rule" in l][0])

In [ ]:
s = run_dag("retrain_dag")

이번에는 점수를 기준 아래로 줍니다.

In [ ]:
s = run_dag("retrain_dag", conf={"score": 0.85})

***확인 포인트***: 고친 뒤에는 0.93 일 때 `deploy` success · `hold` skipped · **`notify` success**, 0.85 일 때 `deploy` skipped · `hold` success · `notify` success 입니다.
어느 갈래로 가든 알림은 갑니다.

**바꿔 보기**
- 점수를 정확히 `0.90` 으로 주면 어느 갈래로 가나요? 코드의 `>=` 를 보고 먼저 예상한 뒤 돌려 봅니다.
- 갈래를 셋으로 늘려 봅니다. 0.95 이상이면 `deploy`, 0.90 이상이면 `canary`(일부만 배포), 그 아래는 `hold`.

## 예제 4 · 리포트 발송 — 집계를 기다렸다 합류, 실패하면 알림

유즈케이스 표의 「집계 완료를 기다렸다가 리포트 생성 · 발송」입니다. 수업 8단계의 `start >> [a, b] >> end` 와 같은 모양에 두 가지를 더합니다.

- 매출 집계와 방문 집계가 **둘 다 끝나야** 리포트를 만듭니다(합류).
- 어느 하나라도 실패하면 리포트 대신 **경보 태스크**가 돕니다(`trigger_rule="one_failed"`).

In [ ]:
%%writefile /content/dags/report_dag.py
# report_dag.py — 리포트 발송: 두 집계가 끝나면 리포트, 실패하면 경보
from datetime import datetime

from airflow.sdk import DAG
from airflow.providers.standard.operators.python import PythonOperator


def agg_sales():
    print("매출 집계 끝")
    return 83000


def agg_visits(params):
    if params["fail_visits"]:
        raise RuntimeError("방문 로그 서버에 연결하지 못했습니다(일부러 낸 오류)")
    print("방문 집계 끝")
    return 412


def make_report(ti):
    sales = ti.xcom_pull(task_ids="agg_sales")
    visits = ti.xcom_pull(task_ids="agg_visits")
    text = f"[일일 리포트] 매출 {sales:,}원 · 방문 {visits}명 · 방문당 매출 {sales / visits:,.0f}원"
    print(text)
    return text


def send(ti):
    print("발송 →", ti.xcom_pull(task_ids="make_report"))


def alert():
    print("경보 → 집계가 실패해 오늘 리포트를 보내지 못했습니다")


with DAG(
    dag_id="report_dag",
    start_date=datetime(2026, 1, 1),
    schedule="0 8 * * *",                   # 매일 08:00
    catchup=False,
    params={"fail_visits": False},
    tags=["week8", "usecase"],
) as dag:
    sales = PythonOperator(task_id="agg_sales", python_callable=agg_sales)
    visits = PythonOperator(task_id="agg_visits", python_callable=agg_visits)
    report = PythonOperator(task_id="make_report", python_callable=make_report)
    t_send = PythonOperator(task_id="send", python_callable=send)
    t_alert = PythonOperator(task_id="alert", python_callable=alert, trigger_rule="one_failed")

    [sales, visits] >> report >> t_send
    [sales, visits] >> t_alert

In [ ]:
s = run_dag("report_dag")
graph("report_dag")

In [ ]:
s = run_dag("report_dag", conf={"fail_visits": True})

***확인 포인트***: 정상일 때는 「매출 83,000원 · 방문 412명 · 방문당 매출 201원」 리포트가 발송되고 `alert` 는 skipped 입니다.
방문 집계를 실패시키면 `agg_sales` 는 success 그대로, `agg_visits` failed, `make_report` · `send` 는 upstream_failed, 그리고 **`alert` 만 success** 입니다.

수업 6단계에서 본 「앞이 실패하면 뒤는 시작하지 않는다」에 「실패했을 때만 도는 태스크」를 더한 모양입니다. 실무에서는 `alert` 자리에 메신저 · 메일 발송이 들어갑니다.

**바꿔 보기**
- `alert` 의 `trigger_rule` 을 `"all_done"` 으로 바꾸면 정상일 때도 경보가 갑니다. 왜 그런지 규칙 이름으로 설명해 봅니다.
- 집계를 하나 더(`agg_refunds`) 붙여 셋이 합류하게 만들고, `graph("report_dag")` 로 모양을 확인합니다.

## 예제 5 · TaskFlow — 같은 정산을 `@dag` · `@task` 로

예제 1 을 TaskFlow 방식으로 다시 씁니다. **하는 일은 똑같고** 쓰는 방법만 다릅니다.

| | 예제 1 (Operator) | 예제 5 (TaskFlow) |
|---|---|---|
| 태스크 만들기 | `PythonOperator(task_id=…, python_callable=…)` | 함수 위에 `@task` |
| 값 받기 | `ti.xcom_pull(task_ids="extract")` | 함수 **인자**로 받는다 |
| 순서 정하기 | `t1 >> t2 >> t3` | 함수를 **부르는 순서**가 곧 의존성 |

In [ ]:
%%writefile /content/dags/settle_taskflow.py
# settle_taskflow.py — 예제 1 과 같은 정산을 TaskFlow 로
import csv, json
from datetime import datetime
from pathlib import Path

from airflow.sdk import dag, task

DATA = Path("/content/data")


@dag(
    dag_id="settle_taskflow",
    start_date=datetime(2026, 1, 1),
    schedule="0 2 * * *",
    catchup=False,
    tags=["week8", "usecase"],
)
def settle_taskflow():

    @task
    def extract() -> int:
        rows = [(i, 1000 * (i % 7 + 1)) for i in range(1, 21)]
        with open(DATA / "orders_tf.csv", "w", newline="") as f:
            csv.writer(f).writerows([("order_id", "amount"), *rows])
        print(f"주문 {len(rows)}건을 받았습니다")
        return len(rows)

    @task
    def summarize(expected: int) -> dict:
        amounts = [int(r["amount"]) for r in csv.DictReader(open(DATA / "orders_tf.csv"))]
        print(f"건수 {expected}건 · 합계 {sum(amounts):,}원")
        return {"count": expected, "total": sum(amounts)}

    @task
    def load(result: dict):
        (DATA / "settle_tf_result.json").write_text(json.dumps(result))
        print("저장 →", result)

    load(summarize(extract()))              # 부르는 순서가 extract → summarize → load


settle_taskflow()                           # 이 줄이 없으면 DAG 가 등록되지 않습니다

In [ ]:
s = run_dag("settle_taskflow")
graph("settle_taskflow")

***확인 포인트***: 그래프가 예제 1 과 같은 `extract → summarize → load` 한 줄이고, 합계도 같은 83,000원입니다. `xcom_pull` 도 `>>` 도 쓰지 않았는데 값이 넘어가고 순서가 생겼습니다.

**바꿔 보기**
- DAG 이름을 `settle_taskflow2` 로 바꾸고 맨 아래 `settle_taskflow()` 줄을 지운 뒤 `run_dag("settle_taskflow2")` 를 실행해 봅니다. 파일에 문법 오류가 없어 `af("dags list-import-errors")` 에도 나오지 않는데 DAG 는 등록되지 않습니다 — 「DAG 가 안 보인다」의 흔한 원인입니다. 확인한 뒤 되돌립니다.
- 두 방식은 한 DAG 안에서 섞어 쓸 수 있습니다. 파이썬 함수가 중심이면 TaskFlow, 셸 명령 · 센서처럼 기성 Operator 를 엮을 때는 Operator 방식이 읽기 쉽습니다.

## 예제 6 · 스케줄 — 크론 표기와 다음 실행 시각

크론 표기는 `분 시 일 월 요일` 다섯 칸입니다. 표기를 바꾸면 실행 시각이 어떻게 달라지는지 `dags next-execution` 으로 여섯 번 치를 직접 봅니다.

| 표기 | 뜻 |
|---|---|
| `@daily` | 매일 00:00 (`0 0 * * *` 과 같음) |
| `30 6 * * *` | 매일 06:30 |
| `0 9 * * 1-5` | 평일 09:00 |
| `*/15 * * * *` | 15분마다 |

In [ ]:
%%writefile /content/dags/schedule_dag.py
# schedule_dag.py — 스케줄만 다른 DAG 다섯 개
from datetime import datetime

import pendulum
from airflow.sdk import DAG
from airflow.providers.standard.operators.empty import EmptyOperator

SCHEDULES = {
    "sch_daily": "@daily",
    "sch_0630": "30 6 * * *",
    "sch_weekday": "0 9 * * 1-5",
    "sch_15min": "*/15 * * * *",
}
for name, cron in SCHEDULES.items():
    with DAG(dag_id=name, start_date=datetime(2026, 1, 1), schedule=cron, catchup=False, tags=["week8"]) as d:
        EmptyOperator(task_id="noop")
    globals()[name] = d

# 같은 "30 6 * * *" 인데 시작 시각에 한국 시간대를 붙인 것
with DAG(dag_id="sch_0630_kst", start_date=pendulum.datetime(2026, 1, 1, tz="Asia/Seoul"),
         schedule="30 6 * * *", catchup=False, tags=["week8"]) as sch_0630_kst:
    EmptyOperator(task_id="noop")

In [ ]:
af("dags reserialize", show=False)          # 새 파일을 지금 읽게 합니다
print("지금(한국 시각):", datetime.now().strftime("%Y-%m-%d %H:%M"), "\n")
for name in ["sch_daily", "sch_0630", "sch_weekday", "sch_15min", "sch_0630_kst"]:
    times = af(f"dags next-execution {name} -n 6", show=False).split()
    kst = [datetime.fromisoformat(t).astimezone().strftime("%m-%d(%a) %H:%M") for t in times]
    print(f"{name:13s} UTC {times[0][11:16]} → 한국 시각  " + " | ".join(kst))

***확인 포인트***: `sch_0630` 의 한국 시각이 **06:30 이 아니라 15:30** 으로 나옵니다. Airflow 의 기본 시간대가 UTC 라서, 크론의 06:30 은 UTC 06:30(한국 15:30)입니다.
`start_date` 에 `tz="Asia/Seoul"` 을 붙인 `sch_0630_kst` 는 한국 시각 06:30 으로 나옵니다. 「아침 6시 반에 돌게 했는데 오후에 돈다」는 실무에서 가장 흔한 스케줄 사고입니다.

`sch_daily` 도 같은 이유로 한국 시각 09:00 입니다(UTC 자정). `sch_weekday` 는 토 · 일을 건너뛰고, `sch_15min` 은 15분 간격으로 나옵니다.
각 줄의 첫 시각은 「지금 켜면 바로 도는 가장 가까운 시각」이라 이미 지난 시각일 수 있습니다.

**바꿔 보기**
- `SCHEDULES` 에 「매주 월요일 07:00」과 「매달 1일 00:00」을 더하고 다음 실행 시각 세 개를 확인합니다.
- 수업에서 만든 `rag_pipeline_final` 은 `@daily` 입니다. 한국 시각으로 몇 시에 도나요?

## 예제 7 · catchup — 밀린 날짜가 한꺼번에 도는 것

수업의 모든 DAG 에 `catchup=False` 를 적었습니다. 이 한 줄을 빼면(또는 True 면) 무슨 일이 생기는지 봅니다.
`start_date` 를 **사흘 전**으로 두고 매일 도는 DAG 를 두 개 만듭니다. 다른 것은 `catchup` 뿐입니다.

이 예제만은 스케줄러가 필요합니다(스케줄에 따라 실행을 **만드는** 것이 스케줄러의 일이라서). 먼저 스케줄러를 켭니다.

In [ ]:
def start_airflow():
    """스케줄러까지 켭니다(airflow standalone) — 수업의 docker compose up -d 에 해당합니다. 20~40초 걸립니다."""
    import urllib.request
    global _SA
    if "_SA" in globals() and _SA.poll() is None:
        print("이미 켜져 있습니다")
        return
    _SA = subprocess.Popen("exec airflow standalone > /content/data/standalone.log 2>&1", shell=True,
                           start_new_session=True)      # 딸린 프로세스까지 한 묶음으로
    for _ in range(90):
        try:
            h = json.loads(urllib.request.urlopen("http://localhost:8080/api/v2/monitor/health", timeout=3).read())
            if h["scheduler"]["status"] == "healthy" and h["dag_processor"]["status"] == "healthy":
                print("켜졌습니다 ·", " · ".join(f"{k} {v['status']}" for k, v in h.items()))
                return
        except Exception:
            pass
        time.sleep(2)
    print("3분 안에 켜지지 않았습니다. !tail -20 /content/data/standalone.log 로 원인을 확인해 주세요.")


def stop_airflow():
    """스케줄러를 끕니다 — 수업의 docker compose stop 에 해당합니다."""
    global _SA
    if "_SA" in globals() and _SA.poll() is None:
        import signal
        os.killpg(_SA.pid, signal.SIGTERM)
        try:
            _SA.wait(timeout=30)
        except Exception:
            os.killpg(_SA.pid, signal.SIGKILL)
    print("껐습니다")


print("도구 준비 완료 · start_airflow( ) · stop_airflow( )")

In [ ]:
%%writefile /content/dags/catchup_dag.py
# catchup_dag.py — catchup 만 다른 DAG 두 개 (시작 = 사흘 전 자정)
import pendulum
from airflow.sdk import DAG
from airflow.providers.standard.operators.python import PythonOperator

START = pendulum.today("Asia/Seoul").subtract(days=3)


def work(logical_date):
    print("처리하는 날짜 →", logical_date.in_timezone("Asia/Seoul").to_date_string())


for name, flag in [("catchup_on", True), ("catchup_off", False)]:
    with DAG(dag_id=name, start_date=START, schedule="@daily", catchup=flag,
             max_active_runs=1, tags=["week8"]) as d:
        PythonOperator(task_id="work", python_callable=work)
    globals()[name] = d

In [ ]:
start_airflow()
af("dags reserialize", show=False)
af("dags unpause catchup_on", show=False)       # 켜는 순간 스케줄러가 「밀린 실행」을 계산합니다
af("dags unpause catchup_off", show=False)
print("스케줄러가 실행을 만들고 돌릴 때까지 60초 기다립니다 …")
time.sleep(60)
for name in ["catchup_on", "catchup_off"]:
    runs = json.loads(af(f"dags list-runs {name} -o json", show=False))
    print(f"\n{name} — 실행 {len(runs)}개")
    for r in sorted(runs, key=lambda x: x["logical_date"]):
        day = datetime.fromisoformat(r["logical_date"]).astimezone().strftime("%m-%d %H:%M")
        print(f"   처리 날짜 {day}   상태 {r['state']}")

In [ ]:
af("dags pause catchup_on", show=False)
af("dags pause catchup_off", show=False)
stop_airflow()                              # 다음 예제는 스케줄러 없이 돕니다

***확인 포인트***: `catchup_on` 은 실행이 **4개**(사흘 전 · 이틀 전 · 어제 · 오늘 자정), `catchup_off` 는 가장 최근 것 **1개**만 생깁니다.
아직 running · queued 인 줄이 있으면 조금 뒤 위 칸을 다시 실행해 봅니다(`max_active_runs=1` 이라 하나씩 순서대로 돕니다).

`start_date=datetime(2026, 1, 1)` 인 DAG 에서 `catchup=False` 를 빼고 켜면, 1월 1일부터 오늘까지 **300개가 넘는 실행**이 한꺼번에 줄을 섭니다.
반대로 「지난달 치를 다시 채워야 한다」처럼 밀린 날짜를 일부러 돌려야 할 때는 이 동작이 필요합니다. 그때는 `catchup` 대신 범위를 정해 돌리는 backfill 을 씁니다.

**바꿔 보기**
- `subtract(days=3)` 을 `days=7` 로 바꾸고 DAG 이름도 `catchup_on7` 처럼 바꿔 다시 해 봅니다. 몇 개가 생기나요?
- `work` 가 받는 `logical_date` 는 「지금」이 아니라 **그 실행이 맡은 날짜**입니다. 밀린 날짜를 다시 돌려도 같은 날짜의 데이터를 처리하게 만드는 것이 멱등한 DAG 의 출발점입니다.

## 예제 8 · 재시도와 멱등성 — 다시 돌려도 안전한가

수업 설계표의 「멱등한가」 칸을 실험으로 확인합니다. DAG 하나에 두 가지를 넣었습니다.

- `flaky_api` — 외부 API 처럼 **가끔 실패하는** 태스크입니다. 처음 두 번은 실패하고 세 번째에 성공합니다. `retries=2` 가 살립니다.
- `load_append` 와 `load_overwrite` — 같은 20건을 적재하는데 하나는 **덧붙이고** 하나는 **덮어씁니다**.

DAG 를 **두 번** 돌린 뒤 두 파일의 줄 수를 셉니다.

In [ ]:
%%writefile /content/dags/idem_dag.py
# idem_dag.py — 재시도가 살리는 태스크 + 덧붙이기와 덮어쓰기
from datetime import datetime, timedelta
from pathlib import Path

from airflow.sdk import DAG
from airflow.providers.standard.operators.python import PythonOperator

DATA = Path("/content/data")
ROWS = [f"{i},{1000 * (i % 7 + 1)}" for i in range(1, 21)]


def flaky_api(ti):
    print(f"{ti.try_number}번째 시도")
    if ti.try_number < 3:
        raise ConnectionError(f"응답 없음({ti.try_number}번째 시도 · 일부러 낸 오류)")
    print("응답 받음")


def load_append():
    with open(DATA / "append.csv", "a") as f:            # "a" = 덧붙이기
        f.write("\n".join(ROWS) + "\n")


def load_overwrite():
    with open(DATA / "overwrite.csv", "w") as f:         # "w" = 덮어쓰기
        f.write("\n".join(ROWS) + "\n")


with DAG(
    dag_id="idem_dag",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    default_args={"retries": 2, "retry_delay": timedelta(seconds=5)},
    tags=["week8"],
) as dag:
    api = PythonOperator(task_id="flaky_api", python_callable=flaky_api)
    a = PythonOperator(task_id="load_append", python_callable=load_append)
    b = PythonOperator(task_id="load_overwrite", python_callable=load_overwrite)

    api >> [a, b]

In [ ]:
for f in ("append.csv", "overwrite.csv"):                # 실험을 처음 상태에서 시작
    if os.path.exists(f"/content/data/{f}"):
        os.remove(f"/content/data/{f}")

s = run_dag("idem_dag")

In [ ]:
s = run_dag("idem_dag")                                  # 같은 DAG 를 한 번 더

print()
for f in ("append.csv", "overwrite.csv"):
    print(f"{f:14s} {len(open('/content/data/' + f).read().splitlines())}줄")

***확인 포인트***: `flaky_api` 는 「1번째 시도 … 오류」 「2번째 시도 … 오류」 「3번째 시도 · 응답 받음」을 거쳐 **success** 입니다. 재시도 사이에 5초씩 쉽니다.
두 번 돌린 뒤 `append.csv` 는 **40줄**, `overwrite.csv` 는 **20줄**입니다. 주문은 20건인데 덧붙인 쪽은 두 배가 됐습니다.

재시도 · 수동 재실행 · catchup 은 모두 「같은 일을 다시 한다」입니다. 덧붙이는 태스크는 다시 돌 때마다 데이터를 망가뜨리고, 덮어쓰는 태스크는 몇 번을 돌려도 같습니다.
8-3 의 `collect`(같은 키에 덮어씀) · `build_index`(문서마다 지우고 다시 넣기)가 그렇게 만든 것입니다.

**바꿔 보기**
- `default_args` 의 `retries` 를 `1` 로 줄여 돌려 봅니다. `flaky_api` 는 어떤 상태로 끝나고, 뒤의 두 태스크는 어떻게 되나요?
- `load_append` 를 멱등하게 고쳐 봅니다. 덧붙이기를 유지하면서도 두 번 돌려 20줄이 되게 하려면 무엇을 먼저 해야 하나요? (힌트: 넣기 전에 같은 주문 번호를 지운다 — 8-3 의 `build_index` 와 같은 방법입니다.)

## 정리 — 내 유즈케이스로 옮기기

여덟 예제에서 쓴 것을 수업의 분해 설계표에 그대로 옮길 수 있습니다.

| 설계할 때 정할 것 | 쓰는 것 | 예제 |
|---|---|---|
| 태스크끼리 무엇을 넘기나 | `return` 과 `xcom_pull`(작은 값만) | 1 · 5 |
| 나쁜 데이터를 어디서 막나 | `raise` 로 실패 / ShortCircuit 으로 건너뜀 | 2 |
| 조건에 따라 갈라지나 | `BranchPythonOperator` + 합류 태스크의 `trigger_rule` | 3 |
| 실패하면 누가 아나 | `trigger_rule="one_failed"` 경보 태스크 | 4 |
| 언제 도나 | 크론 표기 + 시간대 | 6 |
| 밀린 날짜를 어떻게 하나 | `catchup` | 7 |
| 다시 돌려도 안전한가 | `retries` + 덮어쓰기 | 8 |

아래 칸에 내 업무의 반복 작업 하나를 골라 DAG 로 적어 봅니다. 태스크 이름과 화살표만 바꿔도 됩니다.

In [ ]:
%%writefile /content/dags/my_usecase.py
# my_usecase.py — 내 유즈케이스 (태스크 이름 · 순서 · 스케줄을 바꿔 보세요)
from datetime import datetime

from airflow.sdk import DAG
from airflow.providers.standard.operators.python import PythonOperator

STEPS = ["collect", "validate", "process", "report"]      # ← 내 업무의 단계 이름으로


def make(name):
    def _run():
        print(f"{name} — 여기에 할 일을 적습니다")
    return _run


with DAG(
    dag_id="my_usecase",
    start_date=datetime(2026, 1, 1),
    schedule=None,                          # ← 언제 돌아야 하나요? 예: "0 7 * * 1-5"
    catchup=False,
    tags=["week8", "mine"],
) as dag:
    t = {s: PythonOperator(task_id=s, python_callable=make(s)) for s in STEPS}

    t["collect"] >> t["validate"] >> t["process"] >> t["report"]      # ← 병렬 · 분기로 바꿔 보세요

In [ ]:
s = run_dag("my_usecase")
graph("my_usecase")

## 마무리 — 결과 남기기 (선택)

- 가장 기억에 남는 예제 하나의 **상태 표 화면**을 캡처합니다.
- 「바꿔 보기」에서 직접 고친 것과, 고쳤더니 무엇이 달라졌는지 한두 줄로 적습니다.
- 패들릿 **「8주 · Orchestration · AI Service」** 칸에 `이름 — 8-2 실습 예제` 제목으로 올립니다.

수업 노트: https://data-eng.jobability.co.kr